## Aeronef (NUMPYFILE ring)

In [ ]:
import sys
sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

# from FotR_0219 import FRODO, SAM
datafolder = '/home/airbus/CETACEO_cp_interp/DATA/GATALANI_Aero_nef'
db = FRODO(datafolder, format='NUMPYFILE', initial_parse=True, file = ['db_cyc.npy', 'db_random.npy'])

# db = FRODO('/home/airbus/CETACEO_cp_interp/', format = 'NUMPYFILE', initial_parse=True, file=['case_en_npy.npy'])

#FRODO puede llevar distintos "anillos" (los formatos). En este caso llevamos Aeronef, que está organizado mediante archivos binarios de npy. Tenemos dos, y aunque podríamos mezclarlos, vamos a usar solo uno (db_random.npy)
# initial_parse es siempre recomendable activarlo. Estando en True, hacemos un análisis inicial de los datos y completamos sim_metadata, que nos servirá para navegar mejor por la base.

In [ ]:
# SAM tiene muchas subclases útiles. Una de ellas es DictVisualizer, que sirve básicamente para resumir diccionarios. Después tiene Weapons, HDF5Reader y Gardener, que las explicaré en otro momento.

SAM.DictVisualizer.rich_tree(db.sim_metadata)
#sim_metadata guarda variables, parámetros y distribución de los archivos que has leído. Parecerá una tontería, pero te ayuda a saber qué es lo que tienes. En este caso es sencillo. Cuando la base es propia (como con CODA), hay mucho más guardado.
# Es necesario haber ejecutado db.reader.parse_simulation_dirs()

In [ ]:
SAM.DictVisualizer.rich_tree(db.npy_dict)
# En el formato NUMPYFILE, los datos en bruto se guardan en npy_dict

In [ ]:
# Estos dos métodos se utilizan para definir data_dict. Este diccionario resumen las variables que se van a utilizar para fabricar el Dataset final.
# ATENCIÓN: Estos dos métodos son generales para todos los formatos, por lo que los inputs pueden cambiar de uno a otro.

db.extract_inputs(
    keys_inputs={
        'ptos': 'db_random.npy/Airfoil', #ptos siempre es obligatorio porque guarda la geometría. En lo demás, importa solo la forma del array. Si coincide con la geometría, será el campo de una variable. Si no, será parámetro e indicará condición de la simulación.
        'aoa': 'db_random.npy/Alpha', # Por ejemplo, podríamos haber puesto aquí los coeficientes cL o cD, para relacionar el campo fluido con las fuerzas.
        'vel': 'db_random.npy/Vinf'
    },
    keys_aux={
        
        },
    method_to_sort='concave_hull', # Es clave ordenar los puntos para los plots y los posibles cálculos de gradientes posteriores. Hay varias formas de ordenar: 'centroid', 'kdtree', 'concave_hull'.
    common = ['ptos']
)

db.extract_outputs(
    keys_outputs={
        'cp': 'db_random.npy/Cp',
        'cf': 'db_random.npy/Cf'
    },
)

SAM.DictVisualizer.rich_tree(db.data_dict)

### Cálculo de derivadas (2D)

In [ ]:
# Para añadir cálculos indirectos adicionales se utiliza add_aux
import torch
tensor_ptos = torch.from_numpy(db.data_dict['inputs']['ptos'])
for out in ['cp', 'cf']:
    tensor_out = torch.from_numpy(db.data_dict['outputs'][out])

    tensor_deri1 = torch.empty((tensor_out.shape[0], tensor_out.shape[1], 2))
    tensor_deri2 = torch.empty((tensor_out.shape[0], tensor_out.shape[1], 2))
    for case in range(tensor_out.shape[1]):
        tensor_deri1[:, case, :] = SAM.Weapons.finite_diff_derivative(tensor_ptos, tensor_out[:,case], order = 1)
        tensor_deri2[:, case, :] = SAM.Weapons.finite_diff_derivative(tensor_ptos, tensor_out[:,case], order = 2)   

    mod = torch.sqrt(tensor_deri1[:,:,0]**2 + tensor_deri1[:,:,1]**2).reshape_as(tensor_out).numpy()
    mod2 = torch.sqrt(tensor_deri2[:,:,0]**2 + tensor_deri2[:,:,1]**2).reshape_as(tensor_out).numpy()

    db.sets.add_aux(array_name=f'd{out}_ds', array=mod, notes=f'Modulo de derivada 1 de {out}')
    db.sets.add_aux(array_name=f'd{out}2_ds', array=mod2, notes=f'Modulo de derivada 2 de {out}')
    
SAM.DictVisualizer.rich_tree(db.data_dict)

### Cálculo de clusters

In [ ]:
# Por si quieres calcular clusters en la solución: 
df_data_complete, df_metrics = SAM.Weapons.GMM(
    df_data=db.df_data,
    BIC_study=True,
    groupby=["aoa", "vel"],
    nclusters=2,
    features=["dcp_ds", "dcp2_ds", "dcf_ds", "dcf2_ds"],
    save_pictures=True,
    folder_to_save='./GMM_aeronef/',
    n_components_range=range(1, 8),
    covariance_type="diag",
    max_iter=300,
    random_state=123,
    return_metrics_table=True,
    plot_global_analysis=True,
    verbose = True
)

# No hago mucho incapié en esto todavía, pero tiene su miga.

### Exportar datasets

In [ ]:
# Yo tengo mi propio formato de dataset con diccionarios (jset). Este en concreto se guarda en dict_tensors.
db.sets.create_jset(
    sol='all',
    verbose = True,
    save_path = None
)

In [ ]:
# Por supuesto, está SMEAGOL dando su opinión, aunque de momento solo está para el formato CODA.